# Día 9: JavaScript y Comunicación Full Stack con Flask

Este notebook te guiará desde los conceptos básicos de JavaScript hasta la creación de una aplicación web completa que se comunica con Python (Flask) para controlar hardware simulado (como un LED en una Raspberry Pi).

## 1. Introducción a JavaScript

JavaScript (JS) es el lenguaje de programación que se ejecuta en el navegador. Mientras que HTML estructura y CSS decora, **JavaScript da vida** e interactividad.

### Sintaxis Básica y Variables
En JS moderno, usamos `let` y `const` para declarar variables.

```javascript
let nombre = "Juan"; // Variable que puede cambiar
const pi = 3.1416;   // Constante que NO cambia
```

### El DOM (Document Object Model)
El DOM es la representación del HTML que JS puede ver y modificar.
*   `document.getElementById('mi-id')`: Busca un elemento por su ID.
*   `elemento.textContent`: Cambia el texto de un elemento.
*   `elemento.addEventListener('click', funcion)`: Escucha eventos como clics.


In [14]:
%%html
<!-- Ejemplo 1: Manipulación Básica del DOM -->
<button id="cambiarTextoBtn">Haz clic aquí</button>
<p id="textoMonitor">Texto original...</p>

<script>
    const boton = document.getElementById('cambiarTextoBtn');
    const parrafo = document.getElementById('textoMonitor');

    function cambiarMensaje() {
        
            parrafo.textContent = "¡Has vuelto a presionar el botón!";
            parrafo.style.color = "blue";
       
    }

    boton.addEventListener('click', cambiarMensaje);
</script>


## 2. Comunicación con el Servidor: Fetch API

Para que una página web hable con un servidor (por ejemplo, para pedir datos o encender un LED), usamos `fetch()`. Esto es **asíncrono**: la página no se congela mientras espera la respuesta.

### Async / Await
Son palabras clave que hacen que el código asíncrono sea fácil de leer.
*   `async`: Indica que una función tarda un tiempo en completarse.
*   `await`: Pausa la ejecución (dentro de la función) hasta que la promesa se resuelve.

```javascript
async function pedirDatos() {
    try {
        const respuesta = await fetch('/api/datos'); // Hace la petición
        const datos = await respuesta.json(); // Convierte respuesta a JSON
        console.log(datos);
    } catch (error) {
        console.error("Error:", error);
    }
}
```


## 3. Backend con Python: Flask

Flask es un "micro-framework" de Python que nos permite crear servidores web fácilmente.

### Estructura Mínima

```python
from flask import Flask, jsonify

app = Flask(__name__)

@app.route('/')
def home():
    return "Hola Mundo desde Python!"

# Ruta API
@app.route('/api/saludo')
def saludo():
    return jsonify({"mensaje": "Hola desde la API", "status": "ok"})

if __name__ == '__main__':
    app.run(debug=True)
```


## 4. Proyecto Integrador: Panel de Control de Hardware

Vamos a simular un sistema donde una página web controla un LED conectado a una Raspberry Pi.

### Arquitectura
1.  **Frontend (Navegador):** Botón "Encender/Apagar" que envía una petición `POST` al servidor.
2.  **Backend (Python/Flask):** Recibe la petición, actualiza el estado del LED (simulado) y responde al frontend.

### Código del Backend (Python)
Copia este código en un archivo llamado `app.py`.


In [10]:
# app.py (Código del Servidor)
from flask import Flask, jsonify, request

app = Flask(__name__)

# Estado simulado del LED (False = Apagado, True = Encendido)
# En una Raspberry Pi real, aquí usarías: import RPi.GPIO as GPIO
led_state = False

@app.route('/')
def index():
    # En un caso real, aquí servirías tu archivo HTML
    return "Servidor de Control de LED Activo"

@app.route('/api/led', methods=['POST'])
def toggle_led():
    global led_state
    led_state = not led_state # Invertir estado
    
    # Aquí iría el código real de Raspberry Pi:
    # GPIO.output(18, GPIO.HIGH if led_state else GPIO.LOW)
    
    status_text = "ENCENDIDO" if led_state else "APAGADO"
    print(f"Comando recibido. LED ahora está: {status_text}")
    
    return jsonify({
        "led_state": led_state, 
        "message": f"LED correctamente {status_text}"
    })

if __name__ == '__main__':
    print("Iniciando servidor...")
    app.run(port=5000)


Iniciando servidor...
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


### Código del Frontend (HTML + JS)
Este código simula la interfaz. Para probarlo realmente, necesitarías ejecutar el servidor Python anterior.


In [ ]:
%%html
<!-- Panel de Control del LED -->
<div style="border: 2px solid #333; padding: 20px; border-radius: 10px; text-align: center; max-width: 300px;">
    <h2>Control de LED</h2>
    <div id="led-indicator" style="width: 50px; height: 50px; background-color: grey; border-radius: 50%; margin: 10px auto;"></div>
    <p id="status-text">Estado: Desconocido</p>
    <button id="toggleLink" style="padding: 10px 20px; font-size: 16px; cursor: pointer;">Interruptor</button>
</div>

<script>
    const ledIndicator = document.getElementById('led-indicator');
    const statusText = document.getElementById('status-text');
    const btn = document.getElementById('toggleLink');

    // Función para actualizar la UI visualmente
    function updateUI(isOn) {
        if (isOn) {
            ledIndicator.style.backgroundColor = "red";
            ledIndicator.style.boxShadow = "0 0 15px red";
            statusText.textContent = "Estado: ENCENDIDO";
        } else {
            ledIndicator.style.backgroundColor = "grey";
            ledIndicator.style.boxShadow = "none";
            statusText.textContent = "Estado: APAGADO";
        }
    }

    // Función que habla con el Backend (Simulada para este notebook)
    async function toggleLedServer() {
        // NOTA: Esto fallará aquí porque no tenemos el servidor Flask corriendo en el puerto 5000
        // pero este es el código EXACTO que usarías.
        
        try {
            /* 
            // Código Real:
            const response = await fetch('http://localhost:5000/api/led', {
                method: 'POST'
            });
            const data = await response.json();
            updateUI(data.led_state); 
            */
            
            // Simulación para fines educativos en el notebook:
            console.log("Enviando petición al servidor...");
            
            // Simulamos un retardo de red
            await new Promise(r => setTimeout(r, 500)); 
            
            // Alternamos "localmente" para ver el efecto
            const currentColor = ledIndicator.style.backgroundColor;
            const newState = (currentColor !== "red");
            updateUI(newState);
            alert("Petición simulada completada. En producción, esto vendría del servidor Python.");

        } catch (error) {
            console.error("Error de conexión:", error);
            statusText.textContent = "Error de Conexión";
        }
    }

    btn.addEventListener('click', toggleLedServer);
</script>
